# 第11章　閾値と運用点の決め方 ― 臨床に合わせて調整する**『本格実装 医療診断支援AI（実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-impl

## 11.3　検証データで、運用点を選ぶ

In [ ]:
import numpy as npfrom sklearn.metrics import roc_curvefpr, tpr, thr = roc_curve(y_val, prob_val)# 感度(tpr) 0.95 以上を満たす中で、偽陽性率(fpr)が最小の点を選ぶok = tpr >= 0.95best = thr[ok][np.argmin(fpr[ok])]print(f"運用閾値: {best:.3f}")

## 決定曲線分析 ― 閾値を「正味の利益」で選ぶ

In [ ]:
def net_benefit(y_true, prob, pt):    thr = pt                                   # 閾値確率＝そのまま判定閾値    pred = prob >= thr    n = len(y_true)    tp = ((pred == 1) & (y_true == 1)).sum()    fp = ((pred == 1) & (y_true == 0)).sum()    return tp/n - (fp/n) * (pt / (1 - pt))# 臨床的に妥当な pt 帯（例 0.05〜0.2）で、モデルが treat all / treat none を上回るか